# VoiceGuard – HuBERT Transformer (Model 2)

DFADD experiment for Google Colab. Select **Runtime → Change runtime type → T4 GPU** before running the code cell. The cell installs dependencies, trains the frozen-encoder HuBERT baseline, evaluates it, and saves model/results.

In [ ]:
# VOICEGUARD - HuBERT TRANSFORMER (MODEL 2) | DFADD | COLAB T4
!pip install -q transformers datasets librosa soundfile accelerate scikit-learn

import os, random
import numpy as np
import pandas as pd
import torch
import librosa
from datasets import load_dataset
from transformers import AutoFeatureExtractor, HubertModel
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

# 1. Reproducibility and GPU
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))
else: print("WARNING: Enable T4 GPU in Runtime > Change runtime type.")

# 2. Load DFADD
print("\\nLoading DFADD...")
ds = load_dataset("SpeechAntiSpoofingBenchmarks/DFADD", split="test")
real_ids = [i for i, row in enumerate(ds) if int(row["label"]) == 0]
fake_ids = [i for i, row in enumerate(ds) if int(row["label"]) == 1]
print("Available real:", len(real_ids), "| fake:", len(fake_ids))

# Balanced subset (up to 755 per class)
rng = np.random.default_rng(SEED)
n_each = min(755, len(real_ids), len(fake_ids))
selected_ids = np.concatenate([
    rng.choice(real_ids, n_each, replace=False),
    rng.choice(fake_ids, n_each, replace=False)
])
rng.shuffle(selected_ids)

# 3. Preprocess to mono, 16 kHz, fixed 5-second inputs
TARGET_SR, MAX_SECONDS = 16000, 5
MAX_LENGTH = TARGET_SR * MAX_SECONDS
audio_data, labels = [], []
print("\\nPreprocessing audio...")
for count, idx in enumerate(selected_ids, start=1):
    row = ds[int(idx)]
    audio = np.asarray(row["audio"]["array"], dtype=np.float32)
    sr = int(row["audio"]["sampling_rate"])
    if audio.ndim == 2:
        audio = audio.mean(axis=0) if audio.shape[0] <= 8 else audio.mean(axis=1)
    if sr != TARGET_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)
    audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)[:MAX_LENGTH]
    if len(audio) < MAX_LENGTH:
        audio = np.pad(audio, (0, MAX_LENGTH-len(audio)))
    audio_data.append(audio.astype(np.float32))
    labels.append(int(row["label"]))
    if count % 100 == 0: print(f"Processed {count}/{len(selected_ids)}")

audio_data = np.stack(audio_data)
labels = np.asarray(labels, dtype=np.int64)
print("Audio:", audio_data.shape, "| labels:", labels.shape, "| class counts:", np.bincount(labels))

# 4. Train/validation/test split (exploratory random split)
X_train, X_temp, y_train, y_temp = train_test_split(
    audio_data, labels, test_size=0.30, random_state=SEED, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)
print("Train:", len(X_train), "| validation:", len(X_val), "| test:", len(X_test))

# 5. HuBERT feature extractor
MODEL_NAME = "facebook/hubert-base-ls960"
print("\\nLoading HuBERT feature extractor...")
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

# 6. Dataset and loaders
class AudioDataset(Dataset):
    def __init__(self, audios, targets):
        self.audios, self.targets = audios, targets
    def __len__(self):
        return len(self.targets)
    def __getitem__(self, index):
        encoded = feature_extractor(
            self.audios[index], sampling_rate=TARGET_SR,
            return_tensors="pt", padding="max_length",
            max_length=MAX_LENGTH, truncation=True
        )
        return {
            "input_values": encoded.input_values.squeeze(0),
            "label": torch.tensor(self.targets[index], dtype=torch.long)
        }

BATCH_SIZE = 4
train_loader = DataLoader(AudioDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(AudioDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(AudioDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# 7. Frozen HuBERT encoder + trainable classification head
class VoiceGuardHuBERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = HubertModel.from_pretrained(MODEL_NAME)
        for p in self.encoder.parameters():
            p.requires_grad = False
        h = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(h, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 2)
        )
    def forward(self, input_values):
        # Freeze encoder computation, but keep classifier gradients enabled.
        with torch.no_grad():
            output = self.encoder(input_values=input_values)
            features = output.last_hidden_state.mean(dim=1)
        return self.classifier(features)

model = VoiceGuardHuBERT().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)

# 8. Train and validate
EPOCHS = 5
best_val_loss, best_state = float("inf"), None
history = []
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0
    for batch in train_loader:
        x = batch["input_values"].to(device)
        y = batch["label"].to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    train_loss = total_train_loss / max(1, len(train_loader))

    model.eval()
    total_val_loss, val_probs, val_targets = 0.0, [], []
    with torch.no_grad():
        for batch in val_loader:
            x = batch["input_values"].to(device)
            y = batch["label"].to(device)
            logits = model(x)
            total_val_loss += criterion(logits, y).item()
            val_probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
            val_targets.extend(y.cpu().numpy())
    val_loss = total_val_loss / max(1, len(val_loader))
    val_preds = (np.asarray(val_probs) >= 0.5).astype(int)
    val_acc = accuracy_score(val_targets, val_preds)
    history.append({"epoch": epoch+1, "train_loss": train_loss, "val_loss": val_loss, "val_accuracy": val_acc})
    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_accuracy={val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)

# 9. Final test evaluation
model.eval()
test_probs, test_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        x = batch["input_values"].to(device)
        y = batch["label"].to(device)
        logits = model(x)
        test_probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        test_targets.extend(y.cpu().numpy())

test_probs, test_targets = np.asarray(test_probs), np.asarray(test_targets)
test_preds = (test_probs >= 0.5).astype(int)
results = pd.DataFrame([{
    "Model": "HuBERT",
    "Accuracy": accuracy_score(test_targets, test_preds),
    "Precision": precision_score(test_targets, test_preds, zero_division=0),
    "Recall": recall_score(test_targets, test_preds, zero_division=0),
    "F1 Score": f1_score(test_targets, test_preds, zero_division=0),
    "ROC-AUC": roc_auc_score(test_targets, test_probs)
}])
print("\\n========== HuBERT TEST RESULTS ==========")
print(results.to_string(index=False))
print("\\nClassification report:")
print(classification_report(test_targets, test_preds, labels=[0,1], target_names=["Real","Fake"], zero_division=0))
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(test_targets, test_preds, labels=[0,1]))

# 10. Save model and result files
out = "/content/VoiceGuard_HuBERT"
os.makedirs(out, exist_ok=True)
torch.save(model.state_dict(), os.path.join(out, "hubert_model.pth"))
results.to_csv(os.path.join(out, "hubert_results.csv"), index=False)
pd.DataFrame(history).to_csv(os.path.join(out, "hubert_training_history.csv"), index=False)
print("\\nSaved model and CSV results in:", out)
print("This is an exploratory random-split experiment on DFADD's evaluation package, not an official benchmark result.")
